In [ ]:
!pip install -q -U "transformers>=4.51" peft accelerate huggingface_hub
!pip uninstall -y -q torchao
!cp $(find /kaggle/input -name tg.py | head -1) /kaggle/working/tg.py
!ls -la /kaggle/working/tg.py

In [ ]:
import glob, os, shutil
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
from huggingface_hub import snapshot_download

MODEL_ID = "Qwen/Qwen3-1.7B"
DATA  = os.path.dirname(glob.glob('/kaggle/input/**/gpt52-gen_filter/harmful_train.jsonl', recursive=True)[0])
SFT   = glob.glob('/kaggle/input/**/runs/sft/final', recursive=True)[0]
MODEL = snapshot_download(MODEL_ID, local_dir="/kaggle/temp/qwen", allow_patterns=["*.json","*.safetensors","*.txt"])
RUN   = "/kaggle/working/runs"
T     = "torchrun --nproc_per_node=2 /kaggle/working/tg.py"

def pack():
    shutil.make_archive("/kaggle/working/turngate_artifacts", "zip", "/kaggle/working", "runs")
    print("zip size MB:", round(os.path.getsize("/kaggle/working/turngate_artifacts.zip") / 1e6, 1))

print(DATA, SFT, MODEL)

In [ ]:
!{T} eval --model {MODEL} --data_dir {DATA} --split test --per_seed 0 --out {RUN}/eval_zeroshot.json

In [ ]:
!{T} prep_rl --model {MODEL} --adapters {SFT} --data_dir {DATA} --per_seed 8 --out {RUN}/rl_train.jsonl

In [ ]:
!{T} train_rl --model {MODEL} --adapters {SFT} --rl_data {RUN}/rl_train.jsonl --epochs 1 --lr 2e-5 --max_tokens 2048 --accum 12 --log_every 10 --out {RUN}/rl

In [ ]:
pack()

In [ ]:
!{T} eval --model {MODEL} --adapters {SFT},{RUN}/rl/final --data_dir {DATA} --split test --per_seed 0 --out {RUN}/eval_turngate.json

In [ ]:
pack()